In [8]:
import pandas as pd
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [ ]:
ground_truth_df = pd.read_csv("data/ground_truth.csv")

In [3]:
ground_truth_df.head()

,question,document
0,I just found this course — is it too late to j...,74eb249bbf
1,Can I still take the course if I discovered it...,74eb249bbf
2,Is it okay to join the course after it already...,74eb249bbf
3,"If I join now, can I still get a certificate s...",74eb249bbf
4,What do I need to do to qualify for the certif...,74eb249bbf


In [5]:
ground_truth=ground_truth_df.to_dict(orient="records")

In [6]:
ground_truth[0]

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()


documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

index = build_index(documents_llm)

In [11]:
index.search("What is the difference between LLM and RAG?",num_results=5)

[{'id': '649c280e6d',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'How should I prepare documents for RAG?',
  'answer': 'Prepare the data so it is clean, structured, and easy to chunk and retrieve.\n\nCommon steps:\n\n- Remove obvious noise such as broken HTML, duplicate text, boilerplate, OCR errors, repeated headers, and repeated footers.\n- Preserve useful context such as titles, section names, dates, page numbers, speaker names, and Q&A structure.\n- Store the result in a structured format that is easy to process. JSON is often convenient, but it is not mandatory.\n- Chunk the documents in a way that keeps related context together.\n- Keep metadata that may help filtering or ranking later.\n\nThe exact format depends on the source data. The goal is not just to make the text shorter, but to make retrieval more accurate.'},
 {'id': '0bed1f48da',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'dotenv is not recognized. What should 

In [12]:
# boosting the question field to give it more weight in the search results

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [23]:
q=ground_truth[0]
q

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [26]:
doc_id = q["document"]
doc_id

'74eb249bbf'

In [ ]:
results=text_search(q["question"])

In [25]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
85384a18e5 == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False


In [27]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [28]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [29]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — is it too late to join now?


[1, 0, 0, 0, 0]

In [30]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]


Where can I find the live stream URL for office hours or workshop sessions before they start?


[1, 0, 0, 0, 0]

In [31]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I go to check the LLM Zoomcamp syllabus and upcoming deadlines?


[1, 0, 0, 0, 0]

In [32]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [33]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [34]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [35]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [36]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [39]:
relevance_total_sample = compute_relevance_total(ground_truth_sample, text_search)
relevance_total_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

565

In [ ]:
relevance_total[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [44]:
14/15

0.9333333333333333

In [45]:
example=relevance_total[:15]

In [46]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [47]:
cnt / len(example)

0.9333333333333333

In [ ]:
# defining hit rate function

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [49]:
hit_rate(relevance_total)

0.8654867256637168

In [50]:
# calculating mean reciprocal rank (MRR)

total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.5

In [51]:
total_score / len(example)

0.8333333333333334

In [52]:
# defining mean reciprocal rank (MRR) function

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [53]:
mrr(example)

0.8333333333333334

In [54]:
# performing evaluation of the search function using the ground truth data

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [55]:
evaluate(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8654867256637168, 'mrr': 0.743569321533923}

In [56]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [58]:
for boost in [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8884955752212389, 'mrr': 0.7921533923303832}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8973451327433628, 'mrr': 0.7953687315634214}


  0%|          | 0/565 [00:00<?, ?it/s]

...

boost=3.0: {'hit_rate': 0.8654867256637168, 'mrr': 0.743569321533923}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8336283185840708, 'mrr': 0.7090265486725662}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8070796460176991, 'mrr': 0.6747492625368728}


In [59]:
# defining a search function that allows for boosting of question, answer, and section fields

def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [60]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [61]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.978761,0.879381
19,2.0,4.0,0.2,0.978761,0.879381
35,5.0,10.0,0.5,0.978761,0.879381
4,1.0,2.0,0.2,0.978761,0.878555
7,1.0,4.0,0.2,0.980531,0.878112
20,2.0,4.0,0.5,0.976991,0.876401
18,2.0,4.0,0.1,0.978761,0.876136
23,2.0,10.0,0.5,0.978761,0.876047
8,1.0,4.0,0.5,0.984071,0.875841
34,5.0,10.0,0.2,0.976991,0.874749


In [ ]:
# defining a search function with best boost values for question, answer, and section fields
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )